# Flood Hazard Score v2


In [ ]:
# Site configuration — transformation/flood_hazard city configs + model defaults
import os
import sys
from pathlib import Path

_HERE = Path.cwd().resolve()
_FLOOD_HAZARD = None
for _candidate in [_HERE, *_HERE.parents]:
    _probe = _candidate / "flood_hazard" if _candidate.name != "flood_hazard" else _candidate
    if (_probe / "site_config.py").is_file() and (_probe / "config" / "sites").is_dir():
        _FLOOD_HAZARD = _probe
        break
if _FLOOD_HAZARD is None:
    raise FileNotFoundError("Could not locate transformation/flood_hazard from notebook cwd")

sys.path.insert(0, str(_FLOOD_HAZARD))
from site_config import configured_path, load_site_config

FLOOD_HAZARD_ROOT = _FLOOD_HAZARD
FLOODS_ROOT = FLOOD_HAZARD_ROOT  # backward-compatible alias

# Set the city here (edit this line). That value wins for interactive runs.
# Use None to fall back to env FLOODS_SITE (default porto_alegre).
SITE_SLUG = "plymouth"  # or: "porto_alegre" | "edina" | "richfield" | "rochester" | "apple_valley" | None
if SITE_SLUG is None:
    SITE_SLUG = os.environ.get("FLOODS_SITE", "porto_alegre")
SITE_CONFIG = load_site_config(SITE_SLUG, FLOOD_HAZARD_ROOT)
SITE_ROOT = SITE_CONFIG["paths_abs"]["site_root"]
INPUT_DIR = SITE_CONFIG["paths_abs"]["data_input"]
INTERMEDIATE_DIR = SITE_CONFIG["paths_abs"]["data_intermediate"]
OUTPUT_DIR = SITE_CONFIG["paths_abs"]["data_output"]
OUT_ROOT = SITE_CONFIG["paths_abs"]["out"]
CACHE_DIR = SITE_CONFIG["paths_abs"]["cache"]
STYLES_DIR = SITE_CONFIG["paths_abs"]["styles"]
OUTPUT_PREFIX = SITE_CONFIG["output_prefix"]
HAZARD_CFG = SITE_CONFIG["hazard"]
IDW_CFG = SITE_CONFIG["idw"]
MODEL_CONFIG_PATH = SITE_CONFIG["model_config_path"]
print(f"Flood hazard site: {SITE_CONFIG['display_name']} ({SITE_SLUG})")
print(f"Config: {SITE_CONFIG['config_path']}")
print(f"Model defaults: {MODEL_CONFIG_PATH}")
print(f"Weights: {HAZARD_CFG.get('weights')}")
print(f"Inputs -> {INPUT_DIR}")


In [21]:
from rasterio.warp import reproject, Resampling
import os
import numpy as np
import rasterio
import geemap
import json
import geopandas as gpd
from rasterio.features import rasterize


## Score Calculation


In [2]:
# Input rasters
input_paths = {
    "aqueduct_norm": str(INPUT_DIR / SITE_CONFIG["layers"]["aqueduct_norm"]),
    "gfd_count_norm": str(INPUT_DIR / SITE_CONFIG["layers"]["gfd_count_norm"]),
    "gfd_observed_once": str(INPUT_DIR / SITE_CONFIG["layers"]["gfd_observed_once"]),
    "gfplain_250m": str(INPUT_DIR / SITE_CONFIG["layers"]["gfplain"]),
    "jrc_norm": str(INPUT_DIR / SITE_CONFIG["layers"]["jrc_norm"]),
}


def read_single_band_raster(path):
    """Read first band as float32, converting nodata to NaN."""
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        nodata = src.nodata
        if nodata is not None:
            arr = np.where(arr == nodata, np.nan, arr)

        meta = {
            "path": path,
            "shape": arr.shape,
            "crs": str(src.crs),
            "transform": src.transform,
            "nodata": nodata,
            "dtype": str(src.dtypes[0]),
            "count": src.count,
        }
    return arr, meta


layers = {}
layer_meta = {}

for name, path in input_paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing input raster: {path}")

    arr, meta = read_single_band_raster(path)
    layers[name] = arr
    layer_meta[name] = meta

print("✅ Loaded input layers:")
for name, meta in layer_meta.items():
    print(f"- {name}: shape={meta['shape']}, crs={meta['crs']}, nodata={meta['nodata']}")


✅ Loaded input layers:
- aqueduct_norm: shape=(38, 33), crs=EPSG:4326, nodata=None
- gfd_count_norm: shape=(1252, 1057), crs=EPSG:4326, nodata=None
- gfd_observed_once: shape=(1252, 1057), crs=EPSG:4326, nodata=None
- gfplain_250m: shape=(151, 128), crs=EPSG:4326, nodata=None
- jrc_norm: shape=(418, 353), crs=EPSG:4326, nodata=None


In [3]:
# Quick compatibility checks before scoring
ref_name = "aqueduct_norm"
ref_shape = layers[ref_name].shape
ref_crs = layer_meta[ref_name]["crs"]
ref_transform = layer_meta[ref_name]["transform"]
ref_res_x = abs(ref_transform.a)
ref_res_y = abs(ref_transform.e)


def approx_deg_to_meters(res_x_deg, res_y_deg, transform, shape):
    """Approximate pixel size in meters for EPSG:4326 grids at raster center latitude."""
    rows, cols = shape
    center_col = cols / 2.0
    center_row = rows / 2.0
    center_x, center_y = transform * (center_col, center_row)
    lat_rad = np.deg2rad(center_y)

    meters_per_deg_lat = 111320.0
    meters_per_deg_lon = 111320.0 * np.cos(lat_rad)

    res_x_m = res_x_deg * meters_per_deg_lon
    res_y_m = res_y_deg * meters_per_deg_lat
    return center_y, abs(res_x_m), abs(res_y_m)


ref_lat, ref_res_x_m, ref_res_y_m = approx_deg_to_meters(
    ref_res_x, ref_res_y, ref_transform, ref_shape
)

print(
    f"Reference layer: {ref_name} | shape={ref_shape} | crs={ref_crs} "
    f"| res_deg=({ref_res_x:.6f}, {ref_res_y:.6f}) "
    f"| res_m~({ref_res_x_m:.1f}, {ref_res_y_m:.1f}) at lat {ref_lat:.4f}"
)

for name in layers:
    same_shape = layers[name].shape == ref_shape
    same_crs = layer_meta[name]["crs"] == ref_crs

    tr = layer_meta[name]["transform"]
    res_x = abs(tr.a)
    res_y = abs(tr.e)
    same_res = np.isclose(res_x, ref_res_x) and np.isclose(res_y, ref_res_y)

    lat, res_x_m, res_y_m = approx_deg_to_meters(res_x, res_y, tr, layers[name].shape)

    valid = np.isfinite(layers[name])
    vmin = float(np.nanmin(layers[name])) if np.any(valid) else np.nan
    vmax = float(np.nanmax(layers[name])) if np.any(valid) else np.nan

    print(
        f"{name:18s} | shape_ok={same_shape} | crs_ok={same_crs} | res_ok={same_res} "
        f"| res_deg=({res_x:.6f}, {res_y:.6f}) | res_m~({res_x_m:.1f}, {res_y_m:.1f}) "
        f"| min={vmin:.4f} | max={vmax:.4f}"
    )


Reference layer: aqueduct_norm | shape=(38, 33) | crs=EPSG:4326 | res_deg=(0.008983, 0.008983) | res_m~(865.1, 1000.0) at lat -30.1025
aqueduct_norm      | shape_ok=True | crs_ok=True | res_ok=True | res_deg=(0.008983, 0.008983) | res_m~(865.1, 1000.0) | min=0.0000 | max=0.5000
gfd_count_norm     | shape_ok=False | crs_ok=True | res_ok=False | res_deg=(0.000269, 0.000269) | res_m~(26.0, 30.0) | min=0.0000 | max=1.0000
gfd_observed_once  | shape_ok=False | crs_ok=True | res_ok=False | res_deg=(0.000269, 0.000269) | res_m~(26.0, 30.0) | min=0.0000 | max=1.0000
gfplain_250m       | shape_ok=False | crs_ok=True | res_ok=False | res_deg=(0.002246, 0.002246) | res_m~(216.3, 250.0) | min=0.0000 | max=1.0000
jrc_norm           | shape_ok=False | crs_ok=True | res_ok=False | res_deg=(0.000808, 0.000808) | res_m~(77.9, 90.0) | min=0.0000 | max=1.0000


**Score base (MVP, balanced weights):**

```text
flood_score = clamp01( sum(w_i * layer_i) / sum(w_i present) )
```

Weights: JRC 0.45, GFD count 0.30, Aqueduct 0.15, GFPLAIN 0.10.

**Coverage rule (partial score, primary):**
- Compute where **>= 3 of 4** layers are valid.
- Require at least one **fluvial** layer: `jrc_norm` or `aqueduct_norm`.
- Renormalize weights using only available layers at each pixel.

**Strict score (comparison):** all 4 layers must be valid (previous AND mask).


In [4]:
# Align all layers to a common scoring grid (gfplain_250m reference ~250m)
score_ref = HAZARD_CFG.get("reference_grid", "gfplain_250m")
ref_meta = layer_meta[score_ref]
ref_arr = layers[score_ref]


def reproject_to_reference(src_arr, src_meta, ref_meta, ref_shape, resampling):
    dst = np.full(ref_shape, np.nan, dtype="float32")
    reproject(
        source=src_arr,
        destination=dst,
        src_transform=src_meta["transform"],
        src_crs=src_meta["crs"],
        dst_transform=ref_meta["transform"],
        dst_crs=ref_meta["crs"],
        src_nodata=np.nan,
        dst_nodata=np.nan,
        resampling=resampling,
    )
    return dst


aligned = {
    score_ref: ref_arr.copy()
}

for name, arr in layers.items():
    if name == score_ref:
        continue

    is_binary = name in {"gfplain_250m", "gfd_observed_once"}
    rs = Resampling.nearest if is_binary else Resampling.bilinear

    aligned[name] = reproject_to_reference(
        src_arr=arr,
        src_meta=layer_meta[name],
        ref_meta=ref_meta,
        ref_shape=ref_arr.shape,
        resampling=rs,
    )

print("✅ Layers aligned to reference grid:", score_ref)
for name, arr in aligned.items():
    valid = np.isfinite(arr)
    vmin = float(np.nanmin(arr)) if np.any(valid) else np.nan
    vmax = float(np.nanmax(arr)) if np.any(valid) else np.nan
    print(f"- {name:18s} shape={arr.shape} min={vmin:.4f} max={vmax:.4f}")


✅ Layers aligned to reference grid: gfplain_250m
- gfplain_250m       shape=(151, 128) min=0.0000 max=1.0000
- aqueduct_norm      shape=(151, 128) min=0.0000 max=0.5000
- gfd_count_norm     shape=(151, 128) min=0.0000 max=1.0000
- gfd_observed_once  shape=(151, 128) min=0.0000 max=1.0000
- jrc_norm           shape=(151, 128) min=0.2137 max=1.0000


In [5]:
# Compute balanced flood score (strict vs partial >=3/4 coverage)
# Weights / coverage rules come from models/flood_hazard/config.yaml,
# optionally overridden by transformation/flood_hazard/config/sites/{city}.yaml
weights = dict(HAZARD_CFG["weights"])
layer_keys = list(weights.keys())
w = np.array([weights[k] for k in layer_keys], dtype="float32")[:, None, None]

stack = np.stack([aligned[k] for k in layer_keys], axis=0)
layer_valid = np.isfinite(stack)
n_layers_used = layer_valid.sum(axis=0).astype("float32")

fluvial_layers = HAZARD_CFG.get("fluvial_layers", ["jrc_norm", "aqueduct_norm"])
fluvial_present = np.zeros(stack.shape[1:], dtype=bool)
for _name in fluvial_layers:
    if _name in aligned:
        fluvial_present |= np.isfinite(aligned[_name])
if not HAZARD_CFG.get("require_fluvial_layer", True):
    fluvial_present = np.ones_like(fluvial_present, dtype=bool)
min_layers = int(HAZARD_CFG.get("min_layers", 3))

valid_mask_strict = layer_valid.all(axis=0)
valid_mask_partial = (n_layers_used >= min_layers) & fluvial_present

weighted_sum = np.sum(np.where(layer_valid, stack * w, 0.0), axis=0)
weight_total = np.sum(np.where(layer_valid, w, 0.0), axis=0)
weight_total = np.where(weight_total > 0, weight_total, np.nan)

score_raw = weighted_sum / weight_total

flood_score_strict = np.full_like(score_raw, np.nan, dtype="float32")
flood_score_strict[valid_mask_strict] = score_raw[valid_mask_strict]
flood_score_strict = np.clip(flood_score_strict, 0.0, 1.0)

# Primary operational score for screening (river-adjacent coverage preserved)
flood_score_base = np.full_like(score_raw, np.nan, dtype="float32")
flood_score_base[valid_mask_partial] = score_raw[valid_mask_partial]
flood_score_base = np.clip(flood_score_base, 0.0, 1.0)

n_layers_used_out = np.full_like(n_layers_used, np.nan, dtype="float32")
n_layers_used_out[valid_mask_partial] = n_layers_used[valid_mask_partial]

print("✅ flood_score_base (partial, >=3/4 + fluvial) computed")
print("  Valid pixels:", int(valid_mask_partial.sum()))
print("  Min/Max:", float(np.nanmin(flood_score_base)), float(np.nanmax(flood_score_base)))
print("✅ flood_score_strict (all 4 layers) computed")
print("  Valid pixels:", int(valid_mask_strict.sum()))
print("  Gain vs strict:", int(valid_mask_partial.sum() - valid_mask_strict.sum()), "pixels")

for k in layer_keys:
    print(f"  finite {k}: {int(np.isfinite(aligned[k]).sum())}")

# Output paths (used by save + map cells)
out_dir = str(OUTPUT_DIR)
paths = {
    "partial": str(OUTPUT_DIR / SITE_CONFIG["outputs"]["flood_hazard_score"]),
    "strict": str(OUTPUT_DIR / SITE_CONFIG["outputs"]["flood_hazard_score_strict"]),
    "n_layers": str(OUTPUT_DIR / SITE_CONFIG["outputs"]["flood_hazard_n_layers_used"]),
}


✅ flood_score_base (partial, >=3/4 + fluvial) computed
  Valid pixels: 13894
  Min/Max: 0.0 0.8841102123260498
✅ flood_score_strict (all 4 layers) computed
  Valid pixels: 3154
  Gain vs strict: 10740 pixels
  finite jrc_norm: 3979
  finite gfd_count_norm: 18900
  finite aqueduct_norm: 13264
  finite gfplain_250m: 19328


### IDW gap-fill (distance-capped)

Extiende `flood_score_base` en huecos cercanos a píxeles observados con **IDW** y tope de distancia (`MAX_DIST_M`).
Opcional: `USE_GFPLAIN_MASK=True` limita el fill a plana aluvial / contexto fluvial (más conservador).

- Producto nuevo: `flood_score_idw` (observado + relleno)
- Bandas auxiliares: `is_interpolated`, `interp_distance_m`
- El score base **no se modifica**.


In [ ]:
# IDW with distance cap (requires flood_score_base from cell above)
from scipy.spatial import cKDTree
from rasterio.transform import xy

# --- Parameters from model/site config ---
MAX_DIST_M = float(IDW_CFG.get("max_dist_m", 750.0))
MIN_NEIGHBORS = int(IDW_CFG.get("min_neighbors", 3))
IDW_POWER = float(IDW_CFG.get("power", 2.0))
K_NEIGHBORS = int(IDW_CFG.get("k_neighbors", 32))
USE_GFPLAIN_MASK = bool(IDW_CFG.get("use_gfplain_mask", False))
print(f"IDW params: max_dist_m={MAX_DIST_M}, min_neighbors={MIN_NEIGHBORS}, power={IDW_POWER}, k={K_NEIGHBORS}, gfplain_mask={USE_GFPLAIN_MASK}")

observed = np.isfinite(flood_score_base)
gap_mask = ~observed
if USE_GFPLAIN_MASK:
    gap_mask = gap_mask & (
        (aligned["gfplain_250m"] == 1) | fluvial_present
    )

height, width = flood_score_base.shape
rows, cols = np.indices((height, width))
xs, ys = xy(ref_meta["transform"], rows.ravel(), cols.ravel(), offset="center")
xs = np.asarray(xs, dtype="float64")
ys = np.asarray(ys, dtype="float64")

lat0 = float(np.nanmean(ys))
m_per_deg_lat = 111_320.0
m_per_deg_lon = 111_320.0 * np.cos(np.radians(lat0))
coords_m = np.column_stack([xs * m_per_deg_lon, ys * m_per_deg_lat])

obs_flat_idx = np.flatnonzero(observed.ravel())
gap_flat_idx = np.flatnonzero(gap_mask.ravel())

obs_coords = coords_m[obs_flat_idx]
obs_scores = flood_score_base.ravel()[obs_flat_idx].astype("float64")
obs_nlayers = n_layers_used.ravel()[obs_flat_idx].astype("float64")
obs_nlayers = np.where(obs_nlayers >= 3, obs_nlayers, 3.0)

tree = cKDTree(obs_coords)
k_query = min(K_NEIGHBORS, len(obs_flat_idx))
gap_coords = coords_m[gap_flat_idx]

dists, nn_idx = tree.query(gap_coords, k=k_query, workers=-1)
if k_query == 1:
    dists = dists[:, None]
    nn_idx = nn_idx[:, None]

scores_nn = obs_scores[nn_idx]
nlayers_nn = obs_nlayers[nn_idx]

valid_nn = (dists > 1e-6) & (dists <= MAX_DIST_M)
n_valid = valid_nn.sum(axis=1)

with np.errstate(divide="ignore", invalid="ignore"):
    w = np.where(valid_nn, nlayers_nn / (dists ** IDW_POWER), 0.0)
w_sum = w.sum(axis=1)
filled = np.where(w_sum > 0, (w * scores_nn).sum(axis=1) / w_sum, np.nan)
min_dist = np.where(valid_nn, dists, np.inf).min(axis=1)

ok = (n_valid >= MIN_NEIGHBORS) & np.isfinite(filled)
filled = np.clip(filled, 0.0, 1.0)

flood_score_idw = flood_score_base.copy()
is_interpolated = np.zeros_like(flood_score_base, dtype=bool)
interp_distance_m = np.full_like(flood_score_base, np.nan, dtype="float32")

fill_rows, fill_cols = np.unravel_index(gap_flat_idx[ok], (height, width))
flood_score_idw[fill_rows, fill_cols] = filled[ok].astype("float32")
is_interpolated[fill_rows, fill_cols] = True
interp_distance_m[fill_rows, fill_cols] = min_dist[ok].astype("float32")

paths["idw"] = str(OUTPUT_DIR / SITE_CONFIG["outputs"]["flood_hazard_score_idw"])
paths["is_interp"] = str(OUTPUT_DIR / SITE_CONFIG["outputs"]["flood_hazard_is_interpolated"])
paths["interp_dist"] = str(OUTPUT_DIR / SITE_CONFIG["outputs"]["flood_hazard_interp_distance_m"])

print("✅ IDW gap-fill complete")
print("  Observed pixels:", int(observed.sum()))
print("  Gap candidates:", int(gap_mask.sum()))
print("  Interpolated pixels:", int(is_interpolated.sum()))
print("  Total finite (base):", int(np.isfinite(flood_score_base).sum()))
print("  Total finite (idw):", int(np.isfinite(flood_score_idw).sum()))
print("  Gain:", int(np.isfinite(flood_score_idw).sum() - np.isfinite(flood_score_base).sum()), "pixels")
if is_interpolated.any():
    print(
        "  Interp score min/max:",
        float(flood_score_idw[is_interpolated].min()),
        float(flood_score_idw[is_interpolated].max()),
    )
    print(
        "  Mean interp distance (m):",
        float(np.nanmean(interp_distance_m)),
    )



# Mapa comparativo — base vs IDW-filled

Activa/desactiva capas: **observado**, **IDW-filled**, y **solo píxeles interpolados**.


In [31]:
# Comparison map: flood_score_base vs flood_score_idw
if "paths" not in globals() or "idw" not in paths:
    raise RuntimeError("Run the IDW cell first.")

vis_score = {
    "min": 0,
    "max": 1,
    "palette": ["#440154", "#31688e", "#35b779", "#6ece58", "#fde725"],
}

viz_profile = {
    "driver": "GTiff",
    "height": flood_score_base.shape[0],
    "width": flood_score_base.shape[1],
    "count": 1,
    "dtype": "float32",
    "crs": ref_meta["crs"],
    "transform": ref_meta["transform"],
    "nodata": np.nan,
    "compress": "lzw",
}

# Write rasters for geemap tile serving
for arr, path, desc in [
    (flood_score_base, paths["partial"], "flood_score_base_observed"),
    (flood_score_idw, paths["idw"], "flood_score_idw_filled"),
]:
    with rasterio.open(path, "w", **viz_profile) as dst:
        dst.write(arr.astype("float32"), 1)
        dst.set_band_description(1, desc)

# Interpolated-only layer (NaN elsewhere) for visual diff
score_interp_only = np.where(is_interpolated, flood_score_idw, np.nan).astype("float32")
interp_only_path = str(OUTPUT_DIR / SITE_CONFIG["outputs"]["flood_hazard_score_interpolated_only"])
with rasterio.open(interp_only_path, "w", **viz_profile) as dst:
    dst.write(score_interp_only, 1)
    dst.set_band_description(1, "flood_score_interpolated_pixels_only")

with rasterio.open(paths["partial"]) as src:
    b = src.bounds
    center_lat = (b.bottom + b.top) / 2.0
    center_lon = (b.left + b.right) / 2.0

mp = geemap.Map(center=[center_lat, center_lon], zoom=11)
mp.add_basemap("OpenStreetMap.Mapnik")

layer_base = "Hazard — base (observado)"
layer_idw = "Hazard — IDW-filled"
layer_interp = "Hazard — solo interpolados"

mp.add_raster(
    paths["partial"],
    layer_name=layer_base,
    colormap="viridis",
    vmin=0,
    vmax=1,
    opacity=0.78,
)
mp.add_raster(
    paths["idw"],
    layer_name=layer_idw,
    colormap="viridis",
    vmin=0,
    vmax=1,
    opacity=0.0,  # off by default; toggle to compare
)
mp.add_raster(
    interp_only_path,
    layer_name=layer_interp,
    colormap="plasma",
    vmin=0,
    vmax=1,
    opacity=0.0,
)

# mp.add_colorbar(
#     vis_params=vis_score,
#     label="Susceptibilidad (0–1)",
#     layer_name=layer_base,
#     orientation="horizontal",
#     position="bottomright",
#     transparent_bg=True,
# )

# Quick numeric comparison on overlapping + new pixels
delta = flood_score_idw - flood_score_base
delta_interp = delta[is_interpolated]
print("Comparison summary")
print("  Interpolated pixels:", int(is_interpolated.sum()))
if delta_interp.size:
    print("  Mean abs delta on interp pixels:", float(np.nanmean(np.abs(delta_interp))))
print("  Toggle 'IDW-filled' vs 'base' in layer control.")

mp.add_layer_control()
mp



Comparison summary
  Interpolated pixels: 1500
  Mean abs delta on interp pixels: nan
  Toggle 'IDW-filled' vs 'base' in layer control.


Map(center=[-30.101422276740017, -51.161301218817044], controls=(WidgetControl(options=['position', 'transpare…

In [ ]:
# Save score rasters (run compute cell first for paths + arrays)
out_profile = {
    "driver": "GTiff",
    "height": flood_score_base.shape[0],
    "width": flood_score_base.shape[1],
    "count": 1,
    "dtype": "float32",
    "crs": ref_meta["crs"],
    "transform": ref_meta["transform"],
    "nodata": np.nan,
    "compress": "lzw",
}

os.makedirs(out_dir, exist_ok=True)
for key, arr, desc in [
    ("partial", flood_score_base, "flood_score_partial_ge3of4_fluvial"),
    ("strict", flood_score_strict, "flood_score_strict_all4"),
    ("n_layers", n_layers_used_out, "n_layers_used_ge3of4_fluvial"),
]:
    with rasterio.open(paths[key], "w", **out_profile) as dst:
        dst.write(arr.astype("float32"), 1)
        dst.set_band_description(1, desc)
    print("✅ Saved:", paths[key])

# IDW products (run IDW cell first)
if "flood_score_idw" in globals():
    for key, arr, desc in [
        ("idw", flood_score_idw, "flood_score_idw_distance_capped"),
        ("is_interp", is_interpolated.astype("float32"), "is_interpolated_flag"),
        ("interp_dist", interp_distance_m, "interp_distance_m"),
    ]:
        with rasterio.open(paths[key], "w", **out_profile) as dst:
            dst.write(arr.astype("float32"), 1)
            dst.set_band_description(1, desc)
        print("✅ Saved:", paths[key])


#### Step 2. Convert to COG and Generate Tiles

Publish `flood_hazard_score` for web maps: COG + colorized XYZ tiles + value-encoded tiles (hover lookup).
Requires GDAL CLI (`gdal_translate`, `gdaldem`, `gdal_calc.py`, `gdal2tiles.py`) and `styles/flood_hazard_colors.txt`.


In [ ]:
# Paths (run after score GeoTIFF is saved)
from pathlib import Path
import os
import subprocess
import shutil

PROJECT_ROOT = FLOODS_ROOT
in_tif = OUTPUT_DIR / SITE_CONFIG["outputs"]["flood_hazard_score"]
base = STYLES_DIR
out_dir = OUT_ROOT / "flood_hazard_score"
cog_tif = out_dir / "flood_hazard_score_cog.tif"
colorized_tif = out_dir / "flood_hazard_score_colorized.tif"
colors_txt = base / "flood_hazard_colors.txt"
tiles_dir = out_dir / "tiles_visual"
value_tiles_dir = out_dir / "tiles_values"
value_encoded_tif = out_dir / "flood_hazard_score_value_encoded_rgb.tif"

# Jupyter kernels often do not inherit Homebrew paths.
for extra_path in ["/opt/homebrew/bin", "/usr/local/bin", "/opt/homebrew/opt/gdal/bin", "/usr/local/opt/gdal/bin"]:
    if Path(extra_path).exists() and extra_path not in os.environ.get("PATH", ""):
        os.environ["PATH"] = extra_path + os.pathsep + os.environ.get("PATH", "")


def require_cli(command: str) -> str:
    path = shutil.which(command)
    if path:
        return path
    raise RuntimeError(
        f"{command} not found. Install GDAL with `brew install gdal` or add GDAL to PATH.\n"
        f"Current PATH: {os.environ.get('PATH', '')}"
    )


def gdal2tiles_python(gdal2tiles_path: str) -> str:
    with open(gdal2tiles_path, "r", encoding="utf-8", errors="ignore") as f:
        first_line = f.readline().strip()
    if first_line.startswith("#!"):
        shebang_parts = first_line[2:].split()
        if shebang_parts:
            if shebang_parts[0].endswith("env") and len(shebang_parts) > 1:
                resolved = shutil.which(shebang_parts[1])
                if resolved:
                    return resolved
            return shebang_parts[0]
    return shutil.which("python3") or shutil.which("python")


GDAL_TRANSLATE = require_cli("gdal_translate")
GDALDEM = require_cli("gdaldem")
GDAL_CALC = require_cli("gdal_calc.py")
GDAL2TILES = require_cli("gdal2tiles.py")
GDAL2TILES_PYTHON = gdal2tiles_python(GDAL2TILES)
subprocess.run([GDAL2TILES_PYTHON, "-c", "import numpy"], check=True, capture_output=True)
TILE_ZOOM = str(SITE_CONFIG.get("publish", {}).get("tile_zoom", "8-15"))
print("Tile zoom:", TILE_ZOOM)

out_dir.mkdir(parents=True, exist_ok=True)
if not in_tif.exists():
    raise FileNotFoundError(f"Missing input raster: {in_tif}")
if not colors_txt.exists():
    raise FileNotFoundError(f"Missing color table: {colors_txt}")
print("Input:", in_tif)
print("Output dir:", out_dir)
print("GDAL tools:", {"gdal_translate": GDAL_TRANSLATE, "gdaldem": GDALDEM, "gdal_calc.py": GDAL_CALC, "gdal2tiles.py": GDAL2TILES})


In [ ]:
# Convert to COG (preserve float32 values 0-1)
subprocess.run([
    GDAL_TRANSLATE, str(in_tif), str(cog_tif),
    "-of", "COG",
    "-ot", "Float32",
    "-co", "COMPRESS=DEFLATE",
    "-co", "RESAMPLING=NEAREST",
    "-co", "OVERVIEWS=AUTO",
], check=True)
print("Created COG:", cog_tif)

# Color-relief for visual tiles (0-1 palette)
subprocess.run([
    GDALDEM, "color-relief", str(cog_tif), str(colors_txt), str(colorized_tif),
    "-alpha",
], check=True)
print("Created colorized raster:", colorized_tif)

tiles_dir.mkdir(parents=True, exist_ok=True)
subprocess.run([
    GDAL2TILES,
    "-r", "near",
    "-z", TILE_ZOOM,
    "--xyz",
    "-w", "none",
    str(colorized_tif),
    str(tiles_dir),
], check=True)
print("Visual tiles:", tiles_dir)


In [ ]:
# Value tiles for client-side hover (hazard_score = (R + 256*G + 65536*B) / 10000)
# Encode index as integer thousandths in RGB: value = (R + 256*G + 65536*B) / 10000
base_expr = "numpy.where(numpy.isnan(A), 0, numpy.rint(numpy.clip(A,0,1)*10000)).astype(numpy.int64)"
subprocess.run([
    GDAL_CALC,
    "-A", str(cog_tif),
    "--calc", f"bitwise_and({base_expr},255)",
    "--calc", f"bitwise_and(right_shift({base_expr},8),255)",
    "--calc", f"bitwise_and(right_shift({base_expr},16),255)",
    "--type", "Byte",
    "--NoDataValue", "0",
    "--overwrite",
    "--outfile", str(value_encoded_tif),
], check=True)

value_tiles_dir.mkdir(parents=True, exist_ok=True)
subprocess.run([
    GDAL2TILES,
    "-r", "near",
    "-z", TILE_ZOOM,
    "--xyz",
    "-w", "none",
    str(value_encoded_tif),
    str(value_tiles_dir),
], check=True)
print("Value tiles:", value_tiles_dir)
print("Decode in app: score = (R + 256*G + 65536*B) / 10000")


#### Step 2b. Convert IDW hazard score to COG and Generate Tiles

Publicar `flood_hazard_score_idw_poa.tif` (score base + relleno IDW) con el mismo pipeline GDAL.

Requisito: ejecutar antes la celda **IDW gap-fill** y guardar `sites/<site_slug>/data/output/flood_hazard_score_idw_poa.tif`.


In [ ]:
# Paths — IDW hazard score (run after IDW + Save cells)
from pathlib import Path
import os
import subprocess
import shutil

PROJECT_ROOT = FLOODS_ROOT
in_tif_idw = OUTPUT_DIR / SITE_CONFIG["outputs"]["flood_hazard_score_idw"]
base = STYLES_DIR
out_dir_idw = OUT_ROOT / "flood_hazard_score_idw"
cog_tif_idw = out_dir_idw / "flood_hazard_score_idw_cog.tif"
colorized_tif_idw = out_dir_idw / "flood_hazard_score_idw_colorized.tif"
colors_txt = base / "flood_hazard_colors.txt"
tiles_dir_idw = out_dir_idw / "tiles_visual"
value_tiles_dir_idw = out_dir_idw / "tiles_values"
value_encoded_tif_idw = out_dir_idw / "flood_hazard_score_idw_value_encoded_rgb.tif"

# Jupyter kernels often do not inherit Homebrew paths.
for extra_path in ["/opt/homebrew/bin", "/usr/local/bin", "/opt/homebrew/opt/gdal/bin", "/usr/local/opt/gdal/bin"]:
    if Path(extra_path).exists() and extra_path not in os.environ.get("PATH", ""):
        os.environ["PATH"] = extra_path + os.pathsep + os.environ.get("PATH", "")


def require_cli(command: str) -> str:
    path = shutil.which(command)
    if path:
        return path
    raise RuntimeError(
        f"{command} not found. Install GDAL with `brew install gdal` or add GDAL to PATH.\n"
        f"Current PATH: {os.environ.get('PATH', '')}"
    )


def gdal2tiles_python(gdal2tiles_path: str) -> str:
    with open(gdal2tiles_path, "r", encoding="utf-8", errors="ignore") as f:
        first_line = f.readline().strip()
    if first_line.startswith("#!"):
        shebang_parts = first_line[2:].split()
        if shebang_parts:
            if shebang_parts[0].endswith("env") and len(shebang_parts) > 1:
                resolved = shutil.which(shebang_parts[1])
                if resolved:
                    return resolved
            return shebang_parts[0]
    return shutil.which("python3") or shutil.which("python")


GDAL_TRANSLATE = require_cli("gdal_translate")
GDALDEM = require_cli("gdaldem")
GDAL_CALC = require_cli("gdal_calc.py")
GDAL2TILES = require_cli("gdal2tiles.py")
GDAL2TILES_PYTHON = gdal2tiles_python(GDAL2TILES)
subprocess.run([GDAL2TILES_PYTHON, "-c", "import numpy"], check=True, capture_output=True)
TILE_ZOOM = str(SITE_CONFIG.get("publish", {}).get("tile_zoom", "8-15"))
print("Tile zoom:", TILE_ZOOM)

out_dir_idw.mkdir(parents=True, exist_ok=True)
if not in_tif_idw.exists():
    raise FileNotFoundError(f"Missing IDW raster: {in_tif_idw} — run IDW + Save first")
if not colors_txt.exists():
    raise FileNotFoundError(f"Missing color table: {colors_txt}")
print("Input (IDW):", in_tif_idw)
print("Output dir:", out_dir_idw)
print("GDAL tools:", {"gdal_translate": GDAL_TRANSLATE, "gdaldem": GDALDEM, "gdal_calc.py": GDAL_CALC, "gdal2tiles.py": GDAL2TILES})


In [ ]:
# COG + visual tiles (IDW hazard score)
subprocess.run([
    GDAL_TRANSLATE, str(in_tif_idw), str(cog_tif_idw),
    "-of", "COG",
    "-ot", "Float32",
    "-co", "COMPRESS=DEFLATE",
    "-co", "RESAMPLING=NEAREST",
    "-co", "OVERVIEWS=AUTO",
], check=True)
print("Created COG:", cog_tif_idw)

# Color-relief for visual tiles (0-1 palette)
subprocess.run([
    GDALDEM, "color-relief", str(cog_tif_idw), str(colors_txt), str(colorized_tif_idw),
    "-alpha",
], check=True)
print("Created colorized raster:", colorized_tif_idw)

tiles_dir_idw.mkdir(parents=True, exist_ok=True)
subprocess.run([
    GDAL2TILES,
    "-r", "near",
    "-z", TILE_ZOOM,
    "--xyz",
    "-w", "none",
    str(colorized_tif_idw),
    str(tiles_dir_idw),
], check=True)
print("Visual tiles:", tiles_dir_idw)
print("XYZ template:", tiles_dir_idw / "{z}/{x}/{y}.png")


In [ ]:
# Value tiles for client-side hover (IDW hazard score)
# Decode: score = (R + 256*G + 65536*B) / 10000
base_expr = "numpy.where(numpy.isnan(A), 0, numpy.rint(numpy.clip(A,0,1)*10000)).astype(numpy.int64)"
subprocess.run([
    GDAL_CALC,
    "-A", str(cog_tif_idw),
    "--calc", f"bitwise_and({base_expr},255)",
    "--calc", f"bitwise_and(right_shift({base_expr},8),255)",
    "--calc", f"bitwise_and(right_shift({base_expr},16),255)",
    "--type", "Byte",
    "--NoDataValue", "0",
    "--overwrite",
    "--outfile", str(value_encoded_tif_idw),
], check=True)

value_tiles_dir_idw.mkdir(parents=True, exist_ok=True)
subprocess.run([
    GDAL2TILES,
    "-r", "near",
    "-z", TILE_ZOOM,
    "--xyz",
    "-w", "none",
    str(value_encoded_tif_idw),
    str(value_tiles_dir_idw),
], check=True)
print("Value tiles:", value_tiles_dir_idw)
print("Decode in app: score = (R + 256*G + 65536*B) / 10000")


# Validation 

This section validates `flood_score_base` (partial >=3/4 coverage) against independent references (held out from predictors). Re-run cells after scoring to refresh metrics if weights or coverage rules change.

### Data sources (confirmed in repo)

| Reference | Location | Format | Role |
|-----------|----------|--------|------|
| **SkySat / Planet (May 2024)** | `NBS-Project-Preparation/client/public/sample-data/porto-alegre-flood-2024.json` | GeoJSON (197 polygons, 2024-05-06) | Binary observed footprint (same as app `flood_2024_extent` / `in_flood_2024` label) |
| **Copernicus EMSN194 depth** | [maxwaterdepth_cog.tif](https://geo-test-api.s3.us-east-1.amazonaws.com/copernicus_emsn194/release/v1/2024/porto_alegre/maxwaterdepth_cog.tif) | COG (~30 m, depth in m) | Observed max water depth for severity / rank consistency |

Notes:

- SkySat is **not** a local GeoTIFF in the app; it is served as **vector polygons** and rasterized here to the score grid.
- EMSN194 is cataloged in `geospatial-data/catalog/datasets.yaml` (`copernicus_emsn194`).
- Both references describe the **same May 2024 event family**; report metrics as screening validation, not fully independent probability validation.


In [24]:
# SkySat: local GeoJSON used by NBS Site Explorer / grid pipeline
SKYSAT_GEOJSON = (
    "/Users/admin/Desktop/OEF/NBS-Project-Preparation/client/public/sample-data/"
    "porto-alegre-flood-2024.json"
)

# EMSN194: COG on S3 (same asset as geospatial catalog)
EMSN194_COG_URL = (
    "https://geo-test-api.s3.us-east-1.amazonaws.com/copernicus_emsn194/release/v1/"
    "2024/porto_alegre/maxwaterdepth_cog.tif"
)
EMSN194_COG_PATH = f"/vsicurl/{EMSN194_COG_URL}"

print("Validation sources:")
print(f"- SkySat GeoJSON: {'FOUND' if os.path.exists(SKYSAT_GEOJSON) else 'MISSING'}")
print(f"  {SKYSAT_GEOJSON}")
print(f"- EMSN194 COG URL: {EMSN194_COG_URL}")


Validation sources:
- SkySat GeoJSON: FOUND
  /Users/admin/Desktop/OEF/NBS-Project-Preparation/client/public/sample-data/porto-alegre-flood-2024.json
- EMSN194 COG URL: https://geo-test-api.s3.us-east-1.amazonaws.com/copernicus_emsn194/release/v1/2024/porto_alegre/maxwaterdepth_cog.tif


In [25]:
# Build validation rasters on the scoring grid
validation_aligned = {}

# 1) SkySat footprint: rasterize GeoJSON polygons (1 = observed flooded)
if os.path.exists(SKYSAT_GEOJSON):
    with open(SKYSAT_GEOJSON, "r", encoding="utf-8") as f:
        skysat_doc = json.load(f)

    features = skysat_doc.get("geoJson", skysat_doc).get("features", skysat_doc.get("features", []))
    gdf = gpd.GeoDataFrame.from_features(features, crs="EPSG:4326")

    shapes = [(geom, 1) for geom in gdf.geometry if geom is not None]
    skysat_grid = rasterize(
        shapes,
        out_shape=flood_score_base.shape,
        transform=ref_meta["transform"],
        fill=0,
        dtype="uint8",
        all_touched=True,
    ).astype("float32")

    validation_aligned["skysat_extent"] = skysat_grid
    print(
        f"✅ skysat_extent rasterized | polygons={len(shapes)} | "
        f"flooded_pixels={int(np.sum(skysat_grid > 0))}"
    )
else:
    print("⚠️ SkySat GeoJSON not found.")

# 2) EMSN194 depth: read COG from URL and align to scoring grid
try:
    emsn_arr, emsn_meta = read_single_band_raster(EMSN194_COG_PATH)
    validation_aligned["emsn194_depth"] = reproject_to_reference(
        src_arr=emsn_arr,
        src_meta=emsn_meta,
        ref_meta=ref_meta,
        ref_shape=flood_score_base.shape,
        resampling=Resampling.bilinear,
    )

    valid = np.isfinite(validation_aligned["emsn194_depth"])
    print(
        f"✅ emsn194_depth aligned | min={float(np.nanmin(validation_aligned['emsn194_depth'])):.3f} "
        f"max={float(np.nanmax(validation_aligned['emsn194_depth'])):.3f} m | "
        f"positive_pixels={int(np.sum(valid & (validation_aligned['emsn194_depth'] > 0)))}"
    )
except Exception as exc:
    print(f"⚠️ Could not load EMSN194 COG ({exc}). Check network/GDAL vsicurl support.")

if not validation_aligned:
    print("⚠️ No validation layers loaded.")


✅ skysat_extent rasterized | polygons=197 | flooded_pixels=8995
✅ emsn194_depth aligned | min=0.026 max=7.679 m | positive_pixels=4050


In [26]:
# 1) Footprint agreement vs SkySat/Planet (binary metrics)
# Tune threshold as part of sensitivity analysis (e.g., 0.35-0.60)
score_threshold = 0.45

if "skysat_extent" in validation_aligned:
    y_true = validation_aligned["skysat_extent"]

    # Convert to binary observed extent (assumes flood extent raster >0 is flooded)
    obs_flood = y_true > 0

    # Predicted flood extent from score threshold
    pred_flood = flood_score_base >= score_threshold

    # Evaluate only where both score and observation are valid
    eval_mask = np.isfinite(flood_score_base) & np.isfinite(y_true)

    tp = np.sum(pred_flood[eval_mask] & obs_flood[eval_mask])
    fp = np.sum(pred_flood[eval_mask] & ~obs_flood[eval_mask])
    fn = np.sum(~pred_flood[eval_mask] & obs_flood[eval_mask])
    tn = np.sum(~pred_flood[eval_mask] & ~obs_flood[eval_mask])

    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    f1 = 2 * precision * recall / (precision + recall) if np.isfinite(precision) and np.isfinite(recall) and (precision + recall) > 0 else np.nan
    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else np.nan

    print("SkySat/Planet footprint validation")
    print(f"- Threshold: {score_threshold:.2f}")
    print(f"- TP={tp}, FP={fp}, FN={fn}, TN={tn}")
    print(f"- Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}, IoU={iou:.3f}")
else:
    print("⚠️ skysat_extent not loaded. Skipping footprint validation.")


SkySat/Planet footprint validation
- Threshold: 0.45
- TP=3021, FP=94, FN=1107, TN=9672
- Precision=0.970, Recall=0.732, F1=0.834, IoU=0.716


In [27]:
# 2) Severity consistency vs EMSN194 depth (rank/monotonic agreement)
if "emsn194_depth" in validation_aligned:
    emsn_depth = validation_aligned["emsn194_depth"]

    # Evaluate on valid, positive observed depth only
    mask = np.isfinite(flood_score_base) & np.isfinite(emsn_depth) & (emsn_depth > 0)

    if np.sum(mask) < 10:
        print("⚠️ Not enough valid EMSN194 depth pixels for robust severity validation.")
    else:
        s = flood_score_base[mask]
        d = emsn_depth[mask]

        # Spearman correlation via rank transform (no external dependency)
        s_rank = np.argsort(np.argsort(s)).astype("float64")
        d_rank = np.argsort(np.argsort(d)).astype("float64")
        s_rank = (s_rank - s_rank.mean()) / (s_rank.std() + 1e-9)
        d_rank = (d_rank - d_rank.mean()) / (d_rank.std() + 1e-9)
        spearman = float(np.mean(s_rank * d_rank))

        # Bin-based monotonicity check
        bins = np.array([0.0, 0.25, 0.5, 0.75, 1.0])
        bin_ids = np.digitize(s, bins, right=True)
        print("EMSN194 severity consistency")
        print(f"- Valid depth pixels: {int(np.sum(mask))}")
        print(f"- Spearman(score, depth): {spearman:.3f}")

        for b in range(1, len(bins) + 1):
            m = bin_ids == b
            if np.any(m):
                print(
                    f"  bin {b}: n={int(np.sum(m))}, "
                    f"depth_median={float(np.median(d[m])):.3f}, "
                    f"depth_mean={float(np.mean(d[m])):.3f}"
                )
else:
    print("⚠️ emsn194_depth not loaded. Skipping severity validation.")


EMSN194 severity consistency
- Valid depth pixels: 3902
- Spearman(score, depth): 0.769
  bin 1: n=247, depth_median=0.529, depth_mean=0.815
  bin 2: n=947, depth_median=1.531, depth_mean=1.664
  bin 3: n=1617, depth_median=2.761, depth_mean=2.649
  bin 4: n=169, depth_median=3.800, depth_mean=3.791


# Interactive validation map

Compare on the same grid (~250 m):

1. **Flood hazard score** (balanced ensemble, 0–1)
2. **SkySat observed extent** (binary, May 2024)
3. **EMSN194 max water depth** (m)

Toggle layers in the control panel. Porto Alegre basemap underneath.


In [32]:
# Write aligned validation layers to GeoTIFF for local tile serving (same grid as score)
viz_dir = str(OUTPUT_DIR / "validation_viz")
os.makedirs(viz_dir, exist_ok=True)

skysat_viz_path = os.path.join(viz_dir, "skysat_extent_aligned.tif")
emsn_viz_path = os.path.join(viz_dir, "emsn194_depth_aligned.tif")


def write_aligned_geotiff(arr, path, ref_meta, band_desc):
    profile = {
        "driver": "GTiff",
        "height": arr.shape[0],
        "width": arr.shape[1],
        "count": 1,
        "dtype": "float32",
        "crs": ref_meta["crs"],
        "transform": ref_meta["transform"],
        "nodata": np.nan,
        "compress": "lzw",
    }
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(arr.astype("float32"), 1)
        dst.set_band_description(1, band_desc)


if "skysat_extent" in validation_aligned:
    write_aligned_geotiff(
        validation_aligned["skysat_extent"], skysat_viz_path, ref_meta, "skysat_extent"
    )

if "emsn194_depth" in validation_aligned:
    write_aligned_geotiff(
        validation_aligned["emsn194_depth"], emsn_viz_path, ref_meta, "emsn194_depth_m"
    )

# Map center from score extent (operational product = IDW-filled)
out_dir = str(OUTPUT_DIR)
if "paths" not in globals():
    paths = {
        "partial": str(OUTPUT_DIR / SITE_CONFIG["outputs"]["flood_hazard_score"]),
        "idw": os.path.join(out_dir, "flood_hazard_score_idw_poa.tif"),
    }

score_path = paths.get(
    "idw", os.path.join(out_dir, "flood_hazard_score_idw_poa.tif")
)
if not os.path.exists(score_path):
    raise FileNotFoundError(
        f"IDW score GeoTIFF missing: {score_path}. Run IDW gap-fill + Save first."
    )

with rasterio.open(score_path) as src:
    b = src.bounds
    center_lat = (b.bottom + b.top) / 2.0
    center_lon = (b.left + b.right) / 2.0

m_validation = geemap.Map(center=[center_lat, center_lon], zoom=11)
m_validation.add_basemap("OpenStreetMap.Mapnik")

# 1) Flood score (IDW-filled, operational layer)
m_validation.add_raster(
    score_path,
    layer_name="1. Flood hazard score (IDW-filled, 0-1)",
    colormap="viridis",
    vmin=0,
    vmax=1,
    opacity=0.70,
)

# 2) SkySat observed footprint
if os.path.exists(skysat_viz_path):
    m_validation.add_raster(
        skysat_viz_path,
        layer_name="2. SkySat observed extent (2024-05-06)",
        colormap="reds",
        vmin=0,
        vmax=1,
        opacity=0.55,
    )

# 3) EMSN194 depth (m)
emsn_vmax = 4.0
if os.path.exists(emsn_viz_path):
    emsn_pos = validation_aligned["emsn194_depth"]
    if np.any(emsn_pos > 0):
        emsn_vmax = float(np.nanpercentile(emsn_pos[emsn_pos > 0], 95))
    emsn_vmax = max(emsn_vmax, 0.5)

    m_validation.add_raster(
        emsn_viz_path,
        layer_name="3. EMSN194 depth (m)",
        colormap="blues",
        vmin=0,
        vmax=emsn_vmax,
        opacity=0.60,
    )

m_validation.add_legend(
    title="Layers",
    keys=[
        "Score: viridis 0-1",
        "SkySat: red = flooded",
        f"EMSN194: blue depth 0-{emsn_vmax:.1f}m",
    ],
    colors=["#440154", "#d73027", "#08519c"],
    position="bottomright",
)

m_validation.add_layer_control()
m_validation


Map(center=[-30.101422276740017, -51.161301218817044], controls=(WidgetControl(options=['position', 'transpare…

### Validation reporting guidance

For methodology reporting, include:

- Data sources and confirmation they were held out of predictors.
- Threshold used for binary footprint validation and threshold sensitivity sweep.
- Precision/Recall/F1/IoU against SkySat/Planet footprint.
- Spearman and bin-wise depth summaries against EMSN194.
- Caveats (label uncertainty, spatial resolution mismatch, and event representativeness).
